# Даследванне выпадковага лесу (random forest) на падставе лінейнай рэгрэсіі

Датасэт: https://www.kaggle.com/datasets/vishakhdapat/price-of-used-toyota-corolla-cars

Галоўная задача — прадказваць кошт паводле параметраў машыны.

In [180]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse, r2_score as r2, mean_absolute_percentage_error as mape

In [181]:
df = pd.read_csv("ToyotaCorolla.csv", low_memory=False)

***Аналіз звестак у датасэце (EDA) выдалены дзеля зручнасці. Пры патрэбе гл. hw04.***

# Крок 1. Падтрыхтоўка звестак
### Першасная падрыхтоўка (падрабязна гл. у hw04)

In [182]:
# Ствараем прызнак Usage і выдаляем непатрэбныя калонкі
df["Usage"] = df["KM"] / df["Age_08_04"].replace(0, 1)
cols_to_drop = ["Id", "Model", "Mfg_Month", "Mfg_Year", "Cylinders", "CC", "Mfr_Guarantee", "Guarantee_Period"]
df = df.drop(columns=cols_to_drop)

# One-Hot Encoding
df = pd.get_dummies(df, columns=["Fuel_Type", "Color"], drop_first=True)

### Раздзяленне выбаркі

In [183]:
X = df.drop(columns=["Price"])
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Маштабаванне

In [184]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Адказы на пытанні кроку 1

**Ці зрабілі вы перадапрацоўку звестак для выпадковага лесу? Ці адрознівалася яна ад лінейнай мадэлі?**

Так, базавая перадапрацоўка (выдаленне дублікатаў, стварэнне новых прызнакаў, One-Hot Encoding) засталася такой жа. Маштабаванне зробленае толькі для лінейнай рэгрэсіі, дрэвы да яго не адчувальныя.

**Як менавіта вы раздзялілі выбарку?**

Выбарка была падзелена з дапамогай функцыі `train_test_split` на дзве часткі: 80% дадзеных пайшло на трэніроўку (трэніравальная выбарка), а 20% адведзена для фінальнай праверкі (тэставая выбарка).

**На колькі частак трэба дзяліць выбарку пры выкарыстанні крос-валідацыі?**

Звычайна выкарыстоўваюць 5 або 10 частак (так званая K-Fold крос-валідацыя). Гэта дазваляе максімальна аб'ектыўна ацаніць мадэль, зніжаючы залежнасць ад выпадковага падзелу.

**Ці можна не выкарыстоўваць крос-валідацыю? Як дзяліць выбарку ў такім выпадку?**

Так, можна не выкарыстоўваць. У гэтым выпадку ўжываецца метад адкладзенай выбаркі (Hold-out). Звесткі дзеляцца альбо на дзве часткі (Train / Test), альбо на тры (Train / Validation / Test), дзе Validation патрэбна для падбору гіперпараметраў, а Test — для канчатковай ацэнкі.

# Крок 2. Трэніроўка мадэляў

### Функцыя для вываду метрык

In [185]:
def print_metrics(y_true, y_predicted):
    print(f"R²:   {r2(y_true, y_predicted):.4f}")
    print(f"MAE:  {mae(y_true, y_predicted):.2f}")
    print(f"MSE:  {mse(y_true, y_predicted):.2f}")
    print(f"MAPE: {(mape(y_true, y_predicted)*100):.2f}%\n")

### Адно дрэва рашэнняў

<details>
  <summary>Аптымальная глыбіня была падабраная непасрэдна на падставе MAPE (разгарнуць/згарнуць)</summary>
  <code>
  
    best_mape = 100
    best_depth = 0

    for d in range(1, 1000):
        dt = DecisionTreeRegressor(max_depth=d, random_state=42)
        dt.fit(X_train, y_train)
        y_pred_dt = dt.predict(X_test)
        cur_mape = mape(y_test, y_pred_dt)*100
        if cur_mape < best_mape:
            best_mape = cur_mape
            best_depth = d

    print("Лепшая глыбіня:", best_depth)
  </code>

  Вынікам стала 7; гэтая глыбіня не з'яўляецца найлепшай для лесу, там больш эфектыўна паказвае сябе 10 (розніца ў 0.15%)
</details>

In [200]:
start = time.time()
dt = DecisionTreeRegressor(max_depth=7, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Адно дрэва рашэнняў")
print_metrics(y_test, y_pred_dt)
print(f"Час трэніроўкі: {(time.time() - start):.4f} секунд")

Адно дрэва рашэнняў
R²:   0.8991
MAE:  861.72
MSE:  1345863.30
MAPE: 9.07%

Час трэніроўкі: 0.0134 секунд


### Выпадковы лес

In [201]:
start = time.time()
rf = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Выпадковы лес")
print_metrics(y_test, y_pred_rf)
print(f"Час трэніроўкі: {(time.time() - start):.4f} секунд")

Выпадковы лес
R²:   0.9239
MAE:  764.18
MSE:  1014892.84
MAPE: 8.05%

Час трэніроўкі: 0.1577 секунд


## Адказы на пытанні кроку 2

**Параўнанне хуткасці:**

Адно дрэва рашэнняў навучаецца значна (у 11,8 раз) хутчэй за выпадковы лес. Гэта лагічна, бо выпадковы лес у нашым выпадку стварае ансамбль з 100 асобных дрэў.

**Ці можна дасягнуць аднолькавай або блізкай хуткасці?**

Практычна не, калі мы хочам захаваць перавагі лесу. Можна паскорыць лес, калі зменшыць колькасць дрэў (напрыклад, `n_estimators=10`), абмежаваць іх максімальную глыбіню, або выкарыстаць паралельнае вылічэнне (`n_jobs=-1`), што й было зроблена, але базавае дрэва заўсёды будзе хутчэйшым праз розніцу ў аб'ёме вылічэнняў.

**Параўнанне якасці:**

Выпадковы лес заўсёды паказвае больш высокую якасць і стабільнасць. Адно дрэва моцна схільнае да ператрэніроўкі й вельмі адчувальнае да шуму ў звестках. Лес кампенсуе гэтыя недахопы праз усярэдненне прадказанняў незалежных дрэў.

# Крок 3. Ацэнка метрык, параўнанне з лінейнай рэгрэсіяй

### Лінейная рэгрэсія

In [ ]:
start = time.time()
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("Лінейная рэгрэсія")
print_metrics(y_test, y_pred_lr)
print(f"Час трэніроўкі: {(time.time() - start):.4f} секунд")

Лінейная рэгрэсія
R²:   0.8813
MAE:  862.22
MSE:  1583819.29
MAPE: 8.73%

Час трэніроўкі: 0.0081 секунд


## Адказы на пытанні кроку 3

**Якія метрыкі вы выкарысталі для параўнання мадэляў і чаму?**

* **MAE:** Выдатна падыходзіць для бізнес-інтэрпрэтацыі. У нашым выпадку яна паказвае, на колькі эўра ў сярэднім памыляецца мадэль.
* **MSE:** Выкарыстоўваецца таму, што яна строга "штрафуе" мадэль за вялікія памылкі (выбрасы). Калі мадэль памылілася вельмі моцна на адной машыне, MSE адразу ўзрасце.
* **R²:** Гэта базавая метрыка, якая паказвае, якую долю дысперсіі (разбросу) коштаў машын змагла растлумачыць мадэль. Яна патрэбна, каб зразумець, наколькі мадэль лепшая за "дурную" мадэль (DummyRegressor), якая заўсёды прадказвае проста сярэдні кошт.
* **MAPE:** Таксама добрае пасуе для бізнес-інтэрпрэтацыі. Паказвае, на які працэнт ад кошту машыны памыляецца мадэль.

**На якой частцы выбаркі вы лічылі метрыкі?**

Метрыкі лічыліся на тэставай выбарцы (`X_test` і `y_test`). Калі лічыць метрыкі на трэніровачнай выбарцы, мы атрымаем завышаныя вынікі праз перанавучанне і не даведаемся, як мадэль працуе на новых звестках.

**Якая мадэль па выніку справілася лепш?**

Выпадковы лес справіўся лепш за ўсе іншыя мадэлі (лінейную рэгрэсію і адно дрэва рашэнняў). Ансамблевыя метады выдатна працуюць з таблічнымі дадзенымі і аўтаматычна знаходзяць складаныя нелінейныя сувязі.

**Наколькі добрыя атрымаліся вынікі?**

Вынікі атрымаліся вельмі якаснымі. Памылка ў адсотках (MAPE) стала яшчэ меншай, чым для лінейнай рэгрэсіі, што сведчыць пра высокую дакладнасць алгарытму для рэальнага выкарыстання на аўтарынку.